# 2. Gap & Conflict Analysis Testing

This notebook validates the **Gap Analysis** prompt used by ClearSpec AI.

The objective is to verify that the model correctly identifies:

- Logical contradictions
- Ambiguous or vague requirements
- Missing edge cases
- Business rule conflicts
- Assumptions requiring clarification

The production prompt templates are defined in:

- `backend/prompts.py`
- `GAP_SYSTEM`
- `gap_user_msg(...)`

The generated report should identify the majority of intentionally inserted issues.

In [ ]:
import sys
import asyncio

sys.path.insert(0, "../backend")

from dotenv import load_dotenv

load_dotenv("../backend/.env")

from llm_client import call_llm
from prompts import GAP_SYSTEM, gap_user_msg

In [ ]:
# Stories intentionally contain vague wording,
# contradictions and missing edge cases.

STORIES = """
## User Stories

### Story 1: Fast Lab Results

As a doctor,
I want to view lab results quickly,
so that I can make clinical decisions faster.

Acceptance Criteria:
- Results load fast.

---

### Story 2: Patient Notifications

As a patient,
I want to be notified somehow when results are ready.

Acceptance Criteria:
- Notification is sent eventually.

---

### Story 3: Audit Log

As a compliance officer,
I want every access logged.

Acceptance Criteria:
- Audit logs are retained forever.

---

### Story 4: Data Deletion

As a patient,
I want my data removed immediately after revoking consent.

Acceptance Criteria:
- Removal is instant.
"""

EXISTING_CONTEXT = """
Current system:

• Lab results are synchronized from the LIS every 15 minutes.

• Audit logs are retained for 7 years to satisfy HIPAA requirements.

• Patient data deletion follows a mandatory
30-day soft-delete workflow before permanent removal.
"""

In [ ]:
async def evaluate_gap_prompt():
    print("=" * 80)
    print("Running Gap Analysis Prompt")
    print("=" * 80)

    try:
        report = await call_llm(
            GAP_SYSTEM,
            gap_user_msg(STORIES, EXISTING_CONTEXT)
        )

        print(report)

    except Exception as e:
        print("Error while evaluating prompt:")
        print(e)

    print()
    print("=" * 80)
    print("Gap analysis evaluation complete.")

In [ ]:
asyncio.run(evaluate_gap_prompt())

# Output Evaluation Checklist

Verify the generated report identifies the following issues.

## Contradictions

- [ ] Story 3 ("forever") conflicts with 7-year retention policy
- [ ] Story 4 ("immediately") conflicts with 30-day deletion workflow

---

## Vague Terminology

- [ ] "quickly"
- [ ] "fast"
- [ ] "somehow"
- [ ] "eventually"
- [ ] "immediately"
- [ ] "instant"

---

## Missing Edge Cases

- [ ] Critical value paging
- [ ] Notification opt-out
- [ ] Multi-channel notification fallback
- [ ] Failed delivery handling
- [ ] Access authorization validation

---

## Overall Score

Prompt Version: __________

Issue Detection Accuracy: ____ / 10

Report Quality: ____ / 10

Production Ready:

- [ ] Yes
- [ ] Needs Improvement